In [ ]:
"""
max_limit = 1000 for spot markets
"timezone":"UTC"

"""

In [1]:
import pandas as pd
import numpy as np
import requests

import os, sys
import ccxt

In [2]:
api_url = "https://api.binance.com/api/v3/exchangeInfo"

response = requests.get(api_url)
exchange_info = response.json()

spot_li = [i['symbol'] for i in exchange_info['symbols'] if i['quoteAsset']=='USDT']
    

In [3]:
sys.path.insert(1, f"{os.getcwd()}/../../../Lab3310/Systematic-Sherpa")
from data_handler.binance_handler import BinanceHandler

In [4]:
handler = BinanceHandler()

In [5]:
handler.binance_handler_init('SPOT', '1m', 'select')

Getting data: 2023-12-01 00:00:00, BTCUSDT, 1m
[BTCUSDT] Fetched data from - 1701360000000 - 2023-12-01 00:00:00 to 2023-12-01 16:39:00
[BTCUSDT] Fetched data from - 1701420000000 - 2023-12-01 16:40:00 to 2023-12-02 09:19:00
[BTCUSDT] Fetched data from - 1701480000000 - 2023-12-02 09:20:00 to 2023-12-03 01:59:00
[BTCUSDT] Fetched data from - 1701540000000 - 2023-12-03 02:00:00 to 2023-12-03 18:39:00
[BTCUSDT] Fetched data from - 1701600000000 - 2023-12-03 18:40:00 to 2023-12-04 11:19:00
[BTCUSDT] Fetched data from - 1701660000000 - 2023-12-04 11:20:00 to 2023-12-05 03:59:00
[BTCUSDT] Fetched data from - 1701720000000 - 2023-12-05 04:00:00 to 2023-12-05 20:39:00
[BTCUSDT] Fetched data from - 1701780000000 - 2023-12-05 20:40:00 to 2023-12-06 13:19:00
[BTCUSDT] Fetched data from - 1701840000000 - 2023-12-06 13:20:00 to 2023-12-07 05:59:00
[BTCUSDT] Fetched data from - 1701900000000 - 2023-12-07 06:00:00 to 2023-12-07 22:39:00
[BTCUSDT] Fetched data from - 1701960000000 - 2023-12-07 22:40:

In [30]:
df_ = pd.read_csv(r'C:\Users\a1222\Lab3310\Systematic-Sherpa\data_base\BINANCE\SPOT\1m\BTCUSDT_SPOT_1m.csv')
df_['datetime'] = pd.to_datetime(df_['datetime']).dt.tz_localize('Asia/Taipei').dt.tz_convert('UTC')

In [33]:
df_.set_index('datetime')['2023-12-1':].reset_index()

,datetime,open,high,low,close,volume
0,2023-12-01 00:00:00+00:00,37723.97,37744.72,37722.73,37738.91,34.53633
1,2023-12-01 00:01:00+00:00,37738.91,37738.91,37723.72,37723.72,12.71120
2,2023-12-01 00:02:00+00:00,37723.72,37723.73,37699.57,37699.57,33.22694
3,2023-12-01 00:03:00+00:00,37699.58,37699.58,37699.57,37699.57,6.89939
4,2023-12-01 00:04:00+00:00,37699.57,37699.58,37688.27,37688.35,14.22495
...,...,...,...,...,...,...
30503,2023-12-22 04:23:00+00:00,44045.36,44053.07,44045.02,44045.03,20.21157
30504,2023-12-22 04:24:00+00:00,44045.02,44045.24,44045.02,44045.24,26.62383
30505,2023-12-22 04:25:00+00:00,44045.24,44077.86,44045.23,44069.94,27.70364
30506,2023-12-22 04:26:00+00:00,44069.95,44079.65,44061.10,44074.01,18.59401


In [21]:
df_all

,,open,high,low,close,volume
date,code,,,,,
2023-12-01 00:00:00+00:00,BTCUSDT,37723.97,37744.72,37722.73,37738.91,34.53633
2023-12-01 00:01:00+00:00,BTCUSDT,37738.91,37738.91,37723.72,37723.72,12.71120
2023-12-01 00:02:00+00:00,BTCUSDT,37723.72,37723.73,37699.57,37699.57,33.22694
2023-12-01 00:03:00+00:00,BTCUSDT,37699.58,37699.58,37699.57,37699.57,6.89939
2023-12-01 00:04:00+00:00,BTCUSDT,37699.57,37699.58,37688.27,37688.35,14.22495
...,...,...,...,...,...,...
2023-12-22 04:24:00+00:00,BTCUSDT,44045.02,44045.24,44045.02,44045.24,26.62383
2023-12-22 04:25:00+00:00,BTCUSDT,44045.24,44077.86,44045.23,44069.94,27.70364
2023-12-22 04:26:00+00:00,BTCUSDT,44069.95,44079.65,44061.10,44074.01,18.59401


In [6]:
def fetch_ohlcv_data(exchange, symbol, start, freq, limit=1000):
    ohlcv_list = []
    from_ts = exchange.parse8601(start)

    columns = ['date', 'open', 'high', 'low', 'close', 'volume']

    while True:
        ohlcv = exchange.fetch_ohlcv(symbol, freq, since=from_ts, limit=limit)
        ohlcv_list.extend(ohlcv)
        
        if len(ohlcv) < limit:
            break
        
        from_ts = ohlcv[-1][0]

    df = pd.DataFrame(ohlcv_list, columns=columns)
    df['code'] = symbol
    #df['date'] = pd.to_datetime(df['date'], unit='ms').dt.tz_localize('UTC')

    return df

def fetch_all_ohlcv_data(exchange, symbol_list, freq, start):
    df_li = [fetch_ohlcv_data(exchange, symbol, start, freq).set_index(['date', 'code']) for symbol in symbol_list]
    df_all = pd.concat(df_li, axis='rows').reset_index()
    df_all['date'] = pd.to_datetime(df_all['date'], unit='ms').dt.tz_localize('UTC')
    df_all = df_all.set_index(['date', 'code']).sort_index()
    
    return df_all


In [7]:
binance = ccxt.binance()
freq = '1m'
start = '2023-12-01 00:00:00'
symbol_list = ['BTCUSDT']
df_all = fetch_all_ohlcv_data(binance, symbol_list, freq, start)

In [8]:
df_all

,,open,high,low,close,volume
date,code,,,,,
2023-12-01 00:00:00+00:00,BTCUSDT,37723.97,37744.72,37722.73,37738.91,34.53633
2023-12-01 00:01:00+00:00,BTCUSDT,37738.91,37738.91,37723.72,37723.72,12.71120
2023-12-01 00:02:00+00:00,BTCUSDT,37723.72,37723.73,37699.57,37699.57,33.22694
2023-12-01 00:03:00+00:00,BTCUSDT,37699.58,37699.58,37699.57,37699.57,6.89939
2023-12-01 00:04:00+00:00,BTCUSDT,37699.57,37699.58,37688.27,37688.35,14.22495
...,...,...,...,...,...,...
2023-12-22 04:24:00+00:00,BTCUSDT,44045.02,44045.24,44045.02,44045.24,26.62383
2023-12-22 04:25:00+00:00,BTCUSDT,44045.24,44077.86,44045.23,44069.94,27.70364
2023-12-22 04:26:00+00:00,BTCUSDT,44069.95,44079.65,44061.10,44074.01,18.59401


In [11]:
ex = ccxt.binance()
from_ts = ex.parse8601('2023-12-01 00:00:00')
ohlcv_list = []

symbol = 'BTCUSDT'
freq = '1m'
columns = ['date', 'open', 'high', 'low', 'close', 'volume']
ohlcv = ex.fetch_ohlcv(symbol, freq, since=from_ts, limit=1000)
ohlcv_list.append(ohlcv)
while True:
    from_ts = ohlcv[-1][0]
    new_ohlcv = ex.fetch_ohlcv(symbol, freq, since=from_ts, limit=1000)
    ohlcv.extend(new_ohlcv)
    if len(new_ohlcv)!=1000:
    	break

df = pd.DataFrame(ohlcv, columns=columns)
df['date'] = pd.to_datetime(df['date'], unit='ms').dt.tz_localize('UTC')
df['Code'] = symbol

In [34]:
new_ohlcv

[[1703187000000, 43471.09, 43493.9, 43462.0, 43492.08, 20.92466],
 [1703187060000, 43492.07, 43518.25, 43486.11, 43517.99, 15.34516],
 [1703187120000, 43518.0, 43544.2, 43505.43, 43521.1, 34.87255],
 [1703187180000, 43521.1, 43530.0, 43496.0, 43512.36, 15.97069],
 [1703187240000, 43512.36, 43530.63, 43510.18, 43530.63, 10.53373],
 [1703187300000, 43530.63, 43530.63, 43466.96, 43470.75, 22.841],
 [1703187360000, 43470.76, 43478.28, 43445.3, 43454.39, 24.23682],
 [1703187420000, 43454.4, 43454.89, 43422.85, 43422.85, 26.48789],
 [1703187480000, 43422.85, 43443.99, 43411.1, 43443.98, 23.80702],
 [1703187540000, 43443.99, 43457.24, 43423.68, 43457.23, 23.38478],
 [1703187600000, 43457.23, 43482.18, 43456.0, 43481.99, 15.0014],
 [1703187660000, 43481.99, 43482.02, 43476.9, 43478.0, 9.26048],
 [1703187720000, 43478.01, 43479.08, 43441.1, 43446.0, 14.9279],
 [1703187780000, 43445.99, 43459.32, 43445.99, 43454.39, 9.1492],
 [1703187840000, 43454.39, 43484.0, 43454.39, 43483.99, 4.92547],
 [170

In [12]:
df

,date,open,high,low,close,volume,Code
0,2023-12-01 00:00:00+00:00,37723.97,37744.72,37722.73,37738.91,34.53633,BTCUSDT
1,2023-12-01 00:01:00+00:00,37738.91,37738.91,37723.72,37723.72,12.71120,BTCUSDT
2,2023-12-01 00:02:00+00:00,37723.72,37723.73,37699.57,37699.57,33.22694,BTCUSDT
3,2023-12-01 00:03:00+00:00,37699.58,37699.58,37699.57,37699.57,6.89939,BTCUSDT
4,2023-12-01 00:04:00+00:00,37699.57,37699.58,37688.27,37688.35,14.22495,BTCUSDT
...,...,...,...,...,...,...,...
30535,2023-12-22 04:25:00+00:00,44045.24,44077.86,44045.23,44069.94,27.70364,BTCUSDT
30536,2023-12-22 04:26:00+00:00,44069.95,44079.65,44061.10,44074.01,18.59401,BTCUSDT
30537,2023-12-22 04:27:00+00:00,44074.01,44074.01,44062.00,44066.82,12.53198,BTCUSDT
30538,2023-12-22 04:28:00+00:00,44066.81,44075.00,44062.14,44075.00,10.50742,BTCUSDT


# tail